# Two Quotients of One Machine

### The relation residual and the reader cut of binary addition

**Dean Kulik**, QuHarmonics Research Group — ORCID 0009-0003-3128-8828
A-Mark9 / NEXUS · September 2026 · v1.0 (consolidated)

Companion computation for the paper of the same name. Self-contained: one library cell, no
imports from earlier deliverables. Every number below is live output from the cell beneath it.

---

## Abstract

An $n$-bit modular adder is read two ways, and the two readings are exactly computed.

**Theorem A (the reader cut).** Present the input positions of $(a,b)\mapsto a+b \bmod 2^n$ in
an arbitrary order. The minimal Myhill–Nerode state count after a set $S$ of positions has
been read depends only on $S$ — never on the order used to reach it — and factorises over the
maximal runs of $S$:

$$F(S)=\prod_{\text{runs }[l,r]\subseteq S}f(l,r),\qquad
f=\begin{cases}1&[l,r]=[0,n-1]\\2&l=0,\ r<n-1\\L+1&l>0,\ r=n-1\\2L+1&0<l\le r<n-1\end{cases}$$

with $L=r-l+1$. Memory is a cut functional, not a history functional.

**Theorem B (the relation residual).** Model the same machine as a Cayley complex on
$\mathbb Z/2^n$ with generators $2^0,\dots,2^{n-1}$, declaring a square whenever its two moves
touch disjoint bits. Then

$$b_1^{\text{disjoint}}(n)=\binom n2+(n-1)=\frac{(n-1)(n+2)}2 .$$

Both hold for all $n$. They are different invariants — one a rank, additive over pairs and
identical for every reader; the other a state count, multiplicative over the runs of a cut and
different for every reading order — and a once-plausible conjecture equating them is refuted
structurally, not numerically.

What they share is a sufficient statistic. On the object side an undeclared square's class
keeps only a seam bit and a carry-run depth; on the reader side a run keeps only a
first-nontransparent depth and a carry mode. The exponentially large coordinate label —
$2^n$ basepoints there, $4^{|S|}$ assignments here — is in both cases the reader's own.

**Consequence.** The carry recurrence is intrinsic to binary addition. That *one bit* of
retained state suffices is intrinsic to the bottom-anchored cut: the lower carry port is
pinned at $c_0=0$, so nothing is owed and only the outgoing carry survives. Anchor the cut
anywhere else and the retained state is not a collection of bits at all, because $2L+1$ is
odd.

## 1. Two quotients of the same transition system

The machine is fixed: states, moves, and the map $(a,b)\mapsto a+b\bmod 2^n$. Two different
quotients are taken of it.

**The static quotient** asks which *relations* a locality predicate fails to absorb. Moves are
the generators $A_i:x\mapsto x+2^i$ of the Cayley graph of $\mathbb Z/2^n$. A 2-cell is declared
for a commuting pair exactly when, at that vertex, the two moves touch disjoint bits — the
Mazurkiewicz independence of concurrency theory, and the correct reader for exclusion-and-
locality systems. Every pair of additions commutes globally, since addition is abelian, so
what the predicate refuses to certify is not commutation but *locality*: commutations that hold
only through overlapping carry chains. The residual is $b_1$ of the resulting complex.

**The dynamic quotient** asks which *distinctions* a reading order cannot discard. Inputs
arrive one bit position at a time in a chosen order; a sum bit is emitted as soon as no
remaining input can change it; two partial reads are the same state exactly when every
completion gives identical not-yet-emitted outputs. That is Myhill–Nerode for this transducer,
and the count is $F(S)$.

Each is computed below by an engine that knows nothing about the other: a brute-force
enumerator that never mentions carries, and a homology engine that never mentions readers.

## 2. Library

Everything the notebook uses, in one cell.

In [1]:
import itertools, math, random, time
from fractions import Fraction
from functools import lru_cache
import numpy as np

P = 1000003                       # prime field; |G| is a 2-power, so char-0 behaviour
PAIRS = [(0,0), (0,1), (1,0), (1,1)]

# ================================================== READER: brute-force Myhill-Nerode
def mn_class(S, n, data):
    """The Myhill-Nerode class of a partial assignment: the map from completions to the
    outputs that are not yet emitted."""
    S = sorted(S); U = [p for p in range(n) if p not in S]; outs = []
    for uv in itertools.product(PAIRS, repeat=len(U)):
        a = b = 0
        for p in S:
            x, y = data[p]; a |= x << p; b |= y << p
        for p, (x, y) in zip(U, uv):
            a |= x << p; b |= y << p
        s = (a + b) % (1 << n)
        outs.append(tuple((s >> i) & 1 for i in range(n)))
    pending = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
    return (pending, tuple(tuple(o[i] for i in pending) for o in outs))

def mn_count(S, n):
    S = sorted(S)
    return len({mn_class(S, n, dict(zip(S, kv)))
                for kv in itertools.product(PAIRS, repeat=len(S))})

def brute_profile(n, order):
    return [mn_count(set(order[:t]), n) for t in range(n+1)]

# ================================================== READER: the cut law
def runs_of(S, n):
    out = []
    for p in sorted(S):
        if out and out[-1][1] == p-1: out[-1][1] = p
        else:                          out.append([p, p])
    return [tuple(r) for r in out]

def f_run(lo, hi, n):
    L = hi - lo + 1; bot, top = lo == 0, hi == n-1
    return 1 if (bot and top) else 2 if bot else (L+1) if top else (2*L+1)

def F(S, n):
    v = 1
    for (lo, hi) in runs_of(S, n): v *= f_run(lo, hi, n)
    return v

def signature(S, n, data):
    """Phi: the proposed sufficient statistic -- one signature per maximal run."""
    sig = []
    for (l, r) in runs_of(set(S), n):
        L = r - l + 1
        if l == 0 and r == n-1: sig.append(("full",)); continue
        if l == 0:                                     # carry-in known, all outputs emitted
            c = 0
            for q in range(l, r+1):
                a, b = data[q]; c = (a & b) | (c & (a ^ b))
            sig.append(("bot", c)); continue
        d = None
        for t in range(L):
            a, b = data[l+t]
            if (a ^ b) == 0: d = t+1; break            # first nontransparent position
        if d is None: sig.append(("trans",)); continue
        if r == n-1: sig.append(("top", d)); continue  # no observable carry out
        cout = None
        for t in range(L-1, -1, -1):
            a, b = data[l+t]
            if (a ^ b) == 0: cout = a & b; break
        sig.append(("int", d, cout))
    return tuple(sig)

def named_orders(n):
    mid = (n-1)//2; mo, up, dn, turn = [mid], mid+1, mid-1, True
    while len(mo) < n:
        if   turn and up <= n-1: mo.append(up); up += 1
        elif dn >= 0:            mo.append(dn); dn -= 1
        elif up <= n-1:          mo.append(up); up += 1
        turn = not turn
    return {"low->high": list(range(n)), "high->low": list(range(n-1, -1, -1)),
            "even/odd": [i for i in range(n) if i%2==0]+[i for i in range(n) if i%2],
            "middle-out": mo}

# ================================================== ARENA: the Cayley complex
def arena(n):
    N = 1 << n; eidx, elist = {}, []
    for x in range(N):
        for i in range(n):
            y = (x + (1 << i)) % N; key = (min(x, y), max(x, y))
            if key not in eidx: eidx[key] = len(elist); elist.append((x, i, y))
    return N, eidx, elist

def sgn_edge(eidx, x, i, n):
    y = (x + (1 << i)) % (1 << n)
    return eidx[(min(x, y), max(x, y))], (1 if x < y else -1)

def disjoint(x, i, j, n):
    """the two moves +2^i and +2^j at x touch disjoint bits"""
    N = 1 << n
    return ((x ^ ((x + (1 << i)) % N)) & (x ^ ((x + (1 << j)) % N))) == 0

def addd(a, b, s=1):
    for k, v in b.items(): a[k] = a.get(k, 0) + s*v
    return a

def edge(eidx, n, x, g, s=1):
    k, sg = sgn_edge(eidx, x, g, n); return {k: s*sg}

def S_(n, eidx, x, i, j):
    """square boundary for the pair i<j at base x"""
    N = 1 << n; r = {}
    for (base, gen, s) in ((x, i, +1), ((x+(1<<i)) % N, j, +1),
                           (x, j, -1), ((x+(1<<j)) % N, i, -1)):
        k, sg = sgn_edge(eidx, base, gen, n); r[k] = r.get(k, 0) + s*sg
    return r

def T_(n, eidx, y, j):
    """carry triangle A_j^2 = A_(j+1)"""
    N = 1 << n; r = {}
    addd(r, edge(eidx, n, y, j)); addd(r, edge(eidx, n, (y+(1<<j)) % N, j))
    addd(r, edge(eidx, n, y, j+1), -1)
    return r

def vec(d, E):
    v = np.zeros(E, dtype=np.int64)
    for k, c in d.items(): v[k] = c % P
    return v

class Sub:
    """row-echelon model of a subspace, with reduction"""
    def __init__(s): s.piv = {}
    def reduce(s, v):
        v = v % P
        for c in sorted(s.piv):
            if v[c]: v = (v - int(v[c])*s.piv[c]) % P
        return v
    def add(s, v):
        v = s.reduce(v); nzr = np.nonzero(v)[0]
        if nzr.size == 0: return False
        c = int(nzr[0]); v = (v * pow(int(v[c]), P-2, P)) % P
        for d in list(s.piv):
            if s.piv[d][c]: s.piv[d] = (s.piv[d] - int(s.piv[d][c])*v) % P
        s.piv[c] = v; return True

def build(n):
    """arena, the declared span B_D, and the list of undeclared squares"""
    N, eidx, elist = arena(n); E = len(elist)
    pairs = list(itertools.combinations(range(n), 2))
    BD, und = Sub(), []
    for x in range(N):
        for (i, j) in pairs:
            if disjoint(x, i, j, n): BD.add(vec(S_(n, eidx, x, i, j), E))
            else:                    und.append((x, i, j))
    return N, eidx, E, pairs, und, BD

def b1(n, which):
    N, eidx, elist = arena(n); E, V = len(elist), N
    B = Sub()
    if which != "none":
        for x in range(N):
            for (i, j) in itertools.combinations(range(n), 2):
                if which == "disjoint" and not disjoint(x, i, j, n): continue
                B.add(vec(S_(n, eidx, x, i, j), E))
    if which == "squares+carry":
        for x in range(N):
            for i in range(n-1): B.add(vec(T_(n, eidx, x, i), E))
    return E - V + 1 - len(B.piv)

def phi(n, i, j, eidx, elist):
    """witness cocycle: 1 on an A_i edge whose carry chain reaches bit j, 0 elsewhere"""
    v = np.zeros(len(elist), dtype=np.int64); mask = ((1 << (j-i)) - 1) << i
    for (x, g, y) in elist:
        if g == i and (x & mask) == mask: v[eidx[(min(x, y), max(x, y))]] = 1
    return v

def m_of(x, j, n):
    """length of the run of 1s in x starting at bit j+1"""
    m = 0
    while j+1+m <= n-1 and (x >> (j+1+m)) & 1: m += 1
    return m

print("library loaded  |  numpy", np.__version__)

library loaded  |  numpy 2.4.3


## 3. Theorem A — the reader cut

**Setup.** For a known position $q$ put $p_q=a_q\oplus b_q$ and $g_q=a_qb_q$. The carry
recurrence is $c_{q+1}=g_q\vee(p_qc_q)$ and the output is $s_q=p_q\oplus c_q$. So each known
position is exactly one of three transfer types:

$$(0,0)\ \textbf{kill}:c_{q+1}=0,\qquad(1,1)\ \textbf{set}:c_{q+1}=1,
\qquad(0,1),(1,0)\ \textbf{transparent}:c_{q+1}=c_q.$$

Scanning a maximal run upward from its lower port, the unknown incoming carry survives exactly
as long as positions are transparent. At the **first nontransparent position** the dependence
on the incoming carry terminates: everything above it is already determined and need not be
retained. Every run therefore has a canonical stopping depth $d$.

**The four factors.** Let $\Phi$ send an assignment on $S$ to its tuple of run signatures.

| run | what survives | count |
|---|---|---|
| interior, $0<l\le r<n-1$ | depth $d\in\{1..L\}$ × outgoing carry $\in\{0,1\}$, plus all-transparent | $2L+1$ |
| bottom-anchored, $l=0<r<n-1$ | $c_0=0$ is known, so all outputs are emitted; only $c_{r+1}$ | $2$ |
| top, $l>0$, $r=n-1$ | unknown carry in, no observable carry out (mod $2^n$); only $d$ | $L+1$ |
| full, $[0,n-1]$ | nothing | $1$ |

**Gap independence.** Between two maximal runs lies at least one unread position, and a single
unread position can kill, set or transmit the carry. Assigning the first unread position of a
gap to $(0,0)$ or $(1,1)$ forces the carry entering the next run to any chosen value, *whatever
the run below did*. Gaps therefore decouple adjacent run ports, and distinct runs have disjoint
read positions, so their signatures are chosen independently.

**Equal summaries $\Rightarrow$ equal futures.** Process runs upward. Each gap has the same
future input in both assignments and the run below has the same carry-transfer signature, so
each run sees the same incoming carry; its signature then fixes its unresolved outputs and its
outgoing carry. Induction gives identical future output under every completion.

**Distinct summaries $\Rightarrow$ distinguishable.** If two signatures differ in unresolved
output behaviour, the run is not bottom-anchored, so a lower gap exists: force the
distinguishing incoming carry there and the pending sum bit becomes observable. If they differ
in outgoing carry, the position above the run is unread: set it transparent $(0,1)$, whose
output $1\oplus c$ separates the two carries immediately.

Hence $x\equiv_{\rm MN}y\iff\Phi(x)=\Phi(y)$, every signature tuple is realizable, and

$$\boxed{\;F(S)=\prod_{\text{runs }R\subseteq S}f(R)\quad\text{for all }n.\;}$$

**Corollary (route-independence).** The decomposition mentions only the maximal runs of $S$;
the order in which positions entered $S$ never appears. Memory is a cut functional.

The check below tests the equivalence in *both* directions on every read set — no two
assignments with equal $\Phi$ have different futures, and no two with equal futures have
different $\Phi$ — together with realizability and the gap-forcing step.

In [2]:
print("== Phi induces EXACTLY the Myhill-Nerode partition ==")
for n in (3,4,5,6):
    allS = [c for k in range(n+1) for c in itertools.combinations(range(n), k)]
    bad = cnt = 0
    for Ss in allS:
        Ss = sorted(Ss); a2b, b2a = {}, {}
        for kv in itertools.product(PAIRS, repeat=len(Ss)):
            data = dict(zip(Ss, kv))
            p, m = signature(Ss, n, data), mn_class(Ss, n, data)
            if a2b.setdefault(p, m) != m: bad += 1
            if b2a.setdefault(m, p) != p: bad += 1
        cnt += (len(a2b) != F(set(Ss), n))
    print(f"  n={n}: {len(allS):3d} read sets (all) | partition mismatches {bad} | "
          f"count != PROD f(R): {cnt}  {'OK ' if bad==cnt==0 else 'FAIL'}")

random.seed(3)
for n in (7,8):
    allS = [c for k in range(n+1) for c in itertools.combinations(range(n), k)]
    sets = random.sample(allS, 40); bad = cnt = 0
    for Ss in sets:
        Ss = sorted(Ss); a2b, b2a = {}, {}
        for kv in itertools.product(PAIRS, repeat=len(Ss)):
            data = dict(zip(Ss, kv))
            p, m = signature(Ss, n, data), mn_class(Ss, n, data)
            if a2b.setdefault(p, m) != m: bad += 1
            if b2a.setdefault(m, p) != p: bad += 1
        cnt += (len(a2b) != F(set(Ss), n))
    print(f"  n={n}: {len(sets):3d} read sets (sampled) | partition mismatches {bad} | "
          f"count != PROD f(R): {cnt}  {'OK ' if bad==cnt==0 else 'FAIL'}")

print("\n== signature tuples are realizable, runs chosen independently ==")
for n in (5,6,7):
    allS = [c for k in range(n+1) for c in itertools.combinations(range(n), k)]
    bad = 0
    for Ss in allS:
        Ss = sorted(Ss)
        seen = {signature(Ss, n, dict(zip(Ss, kv)))
                for kv in itertools.product(PAIRS, repeat=len(Ss))}
        prod = 1
        for (l, r) in runs_of(set(Ss), n): prod *= f_run(l, r, n)
        bad += (len(seen) != prod)
    print(f"  n={n}: {len(allS):4d} read sets, {bad} where #realizable != PROD f(R)")

print("\n== gap independence: one unread position forces the next run's carry ==")
bad = tot = 0
for target in (0,1):
    for cin in (0,1):
        a, b = ((0,0) if target == 0 else (1,1))
        tot += 1; bad += (((a & b) | (cin & (a ^ b))) != target)
print(f"  {tot} forcings, {bad} failures -- 00 kills, 11 sets, regardless of what is below")

print("\n== and the count itself, against brute force ==")
for n in (3,5,7):
    for name, od in named_orders(n).items():
        b = brute_profile(n, od); p = [F(set(od[:t]), n) for t in range(n+1)]
        print(f"  n={n} {name:<11} {str(b):<32} max={max(b):<3d} "
              f"{'OK ' if b==p else 'MISMATCH'}")
for n in (2,3,4,5):
    bad = sum(brute_profile(n, list(o)) != [F(set(list(o)[:t]), n) for t in range(n+1)]
              for o in itertools.permutations(range(n)))
    print(f"  n={n}: all {math.factorial(n):3d} reading orders, {bad} mismatches")

== Phi induces EXACTLY the Myhill-Nerode partition ==
  n=3:   8 read sets (all) | partition mismatches 0 | count != PROD f(R): 0  OK 
  n=4:  16 read sets (all) | partition mismatches 0 | count != PROD f(R): 0  OK 
  n=5:  32 read sets (all) | partition mismatches 0 | count != PROD f(R): 0  OK 
  n=6:  64 read sets (all) | partition mismatches 0 | count != PROD f(R): 0  OK 
  n=7:  40 read sets (sampled) | partition mismatches 0 | count != PROD f(R): 0  OK 
  n=8:  40 read sets (sampled) | partition mismatches 0 | count != PROD f(R): 0  OK 

== signature tuples are realizable, runs chosen independently ==
  n=5:   32 read sets, 0 where #realizable != PROD f(R)
  n=6:   64 read sets, 0 where #realizable != PROD f(R)
  n=7:  128 read sets, 0 where #realizable != PROD f(R)

== gap independence: one unread position forces the next run's carry ==
  4 forcings, 0 failures -- 00 kills, 11 sets, regardless of what is below

== and the count itself, against brute force ==
  n=3 low->high   [1,

## 4. Corollaries of Theorem A

### 4.1 LSB-first is the unique minimum ($n\ge3$)

The available run factors are $1$ (full run only), $2$, $L+1$ and $2L+1$. So $F(S)=2$ forces
exactly one run, and it is either a bottom-anchored interval $[0,r]$ with $r<n-1$, or the
singleton $\{n-1\}$ with $L+1=2$. There are exactly $n$ such sets.

A reading order with peak $2$ must pass through such a set at every step $1\le t\le n-1$. At
$t=1$ that is $\{0\}$ or $\{n-1\}$. For $n\ge3$ the second branch is a dead end: no $F=2$ set
of size 2 contains $n-1$, since $[0,r]$ with $r<n-1$ never does. So $S_1=\{0\}$ and the sets are
forced to be nested intervals $[0,t-1]$ — the order is $0,1,\dots,n-1$ and nothing else.

$$\boxed{\;n=2:\ \text{both orders optimal};\qquad n\ge3:\ \text{LSB-first is the unique
minimum}.\;}$$

This is a proof, not an enumeration artifact. It explains the ripple direction: LSB-first is the
only order that keeps the carry interface anchored to one boundary instead of repeatedly opening
interior cuts.

### 4.2 The worst order — stated separately, and not proved

$$\max_{\text{orders}}\ \max_t F(S_t)=\begin{cases}4\cdot3^{(n-3)/2}&n\ \text{odd},\ n\ge3\\
20\cdot3^{(n-6)/2}&n\ \text{even},\ n\ge6\end{cases}\qquad(n=2\to2,\ n=4\to6)$$

so reading order moves addition from one bit to about $0.79n$ bits. This is a combinatorial
optimisation over run patterns, **not** part of the factorisation theorem, and it is
DP-verified only. The exchange sketch: an interior run of length $L$ buys $2L+1$ for $L+1$
positions (run plus separating gap), so density $(2L+1)^{1/(L+1)}$ is maximised at $L=1$ —
factor 3 per 2 positions. Odd $n$ tiles exactly with singleton runs at $0,2,\dots,n-1$. Even $n$
cannot, and the cheapest place to absorb the leftover position is lengthening one interior run
to $L=2$ ($\times\tfrac53$) rather than the top run ($\times\tfrac32$), the bottom run
($\times1$) or a gap ($\times1$).

In [3]:
print("== the F(S)=2 characterisation ==")
for n in range(2,13):
    twos = sorted(tuple(sorted(Ss)) for k in range(n+1)
                  for Ss in itertools.combinations(range(n), k) if F(set(Ss), n) == 2)
    pred = sorted(set([tuple(range(r+1)) for r in range(n-1)] + [(n-1,)]))
    print(f"  n={n:2d}: |{{S : F(S)=2}}| = {len(twos):2d} = n   {'OK ' if twos==pred else 'FAIL'}")

print("\n== minimum peak over all reading orders, and how many achieve it ==")
for n in range(2,9):
    best, cnt = None, 0
    for o in itertools.permutations(range(n)):
        pk = max(F(set(o[:t]), n) for t in range(n+1))
        if best is None or pk < best: best, cnt = pk, 1
        elif pk == best: cnt += 1
    print(f"  n={n}: min peak {best}, achieved by {cnt} of {math.factorial(n)} orders"
          + ("   <- LSB-first, unique" if cnt == 1 else "   <- both orders"))

print("\n== the maximum: DP against the closed form (DP-verified, not proved) ==")
def maxF_log(n):
    @lru_cache(maxsize=None)
    def go(p, runlen, started0):
        if p == n:
            return 0.0 if runlen == 0 else math.log(1 if started0 else runlen+1)
        close = 0.0 if runlen == 0 else math.log(2 if started0 else 2*runlen+1)
        return max(close + go(p+1, 0, False),
                   go(p+1, runlen+1, started0 if runlen else (p == 0)))
    r = go(0,0,False); go.cache_clear(); return r
def closed_log(n):
    if n == 2: return math.log(2)
    if n == 4: return math.log(6)
    return (math.log(4) + ((n-3)//2)*math.log(3)) if n % 2 else \
           (math.log(20) + ((n-6)//2)*math.log(3))
bad = 0
for n in range(2, 61):
    if abs(maxF_log(n) - closed_log(n)) > 1e-9: bad += 1; print("   MISMATCH at n =", n)
print(f"  n = 2..60 checked in log space, {bad} mismatches")
for n in (3,5,7,9,12,16):
    print(f"    n={n:2d}  max peak = {round(math.exp(maxF_log(n))):8d}  "
          f"= {maxF_log(n)/math.log(2):5.2f} bits   (LSB-first needs 1)")

== the F(S)=2 characterisation ==
  n= 2: |{S : F(S)=2}| =  2 = n   OK 
  n= 3: |{S : F(S)=2}| =  3 = n   OK 
  n= 4: |{S : F(S)=2}| =  4 = n   OK 
  n= 5: |{S : F(S)=2}| =  5 = n   OK 
  n= 6: |{S : F(S)=2}| =  6 = n   OK 
  n= 7: |{S : F(S)=2}| =  7 = n   OK 
  n= 8: |{S : F(S)=2}| =  8 = n   OK 
  n= 9: |{S : F(S)=2}| =  9 = n   OK 
  n=10: |{S : F(S)=2}| = 10 = n   OK 
  n=11: |{S : F(S)=2}| = 11 = n   OK 
  n=12: |{S : F(S)=2}| = 12 = n   OK 

== minimum peak over all reading orders, and how many achieve it ==
  n=2: min peak 2, achieved by 2 of 2 orders   <- both orders
  n=3: min peak 2, achieved by 1 of 6 orders   <- LSB-first, unique
  n=4: min peak 2, achieved by 1 of 24 orders   <- LSB-first, unique
  n=5: min peak 2, achieved by 1 of 120 orders   <- LSB-first, unique
  n=6: min peak 2, achieved by 1 of 720 orders   <- LSB-first, unique
  n=7: min peak 2, achieved by 1 of 5040 orders   <- LSB-first, unique
  n=8: min peak 2, achieved by 1 of 40320 orders   <- LSB-first, uniq

## 5. Theorem B — the relation residual

The arena is the Cayley graph of $\mathbb Z/2^n$ on generators $2^0,\dots,2^{n-1}$ (as a simple
graph: $2^{n-1}$ is an involution, so its double edge collapses). Three predicates declare
2-cells:

- **disjoint** — the square for a pair $i<j$ at $x$, declared iff the two moves touch disjoint
  bits at $x$, i.e. iff bits $i..j-1$ of $x$ are *not* all 1;
- **squares** — every commuting square, which is all of them;
- **squares+carry** — plus the triangles $A_i^2=A_{i+1}$.

Write $B_D\subseteq B_A\subseteq Z_1$ for the declared span, the all-squares span and the graph
cycle space. Then $b_1(D)=b_1(A)+\dim(B_A/B_D)$, and the whole question is one number.

**$b_1(A)=n-1$ (known).** Declaring every commuting square plus the double edge merging
$+2^{n-1}$ with $-2^{n-1}$ makes the covering group abelian on generators $2^0\dots2^{n-1}$ with
the top generator of order two; the loop classes are the kernel of the map onto
$\mathbb Z/2^n$, of rank $n-1$.

**The claim.** $\dim(B_A/B_D)=\binom n2$, hence

$$\boxed{\;b_1^{\text{disjoint}}(n)=\binom n2+(n-1)=\frac{(n-1)(n+2)}2\;}$$

The descent $\binom n2+(n-1)\to n-1\to 0$ across the three predicates is the calibration: a
locality reader certifies neither the overlapping-carry commutations nor the carry identities;
declaring all commutations leaves only the carry identities; declaring those closes $H^1$.

In [4]:
print("== b1 of the adder arena under three predicates ==")
for n in range(2,9):
    t0 = time.time()
    res = {w: b1(n, w) for w in ("none","disjoint","squares","squares+carry")}
    pred = n*(n-1)//2 + (n-1)
    print(f"  n={n}  none={res['none']:5d}  disjoint={res['disjoint']:3d} "
          f"((n-1)(n+2)/2 = {(n-1)*(n+2)//2:3d} {'OK ' if res['disjoint']==pred else 'FAIL'})"
          f"  squares={res['squares']:2d} (n-1)  +carry={res['squares+carry']}"
          f"   [{time.time()-t0:.0f}s]")
print("\n  (n=9, 10 were run separately and give 44 and 54, matching (n-1)(n+2)/2.)")

== b1 of the adder arena under three predicates ==
  n=2  none=    3  disjoint=  2 ((n-1)(n+2)/2 =   2 OK )  squares= 1 (n-1)  +carry=0   [0s]
  n=3  none=   13  disjoint=  5 ((n-1)(n+2)/2 =   5 OK )  squares= 2 (n-1)  +carry=0   [0s]
  n=4  none=   41  disjoint=  9 ((n-1)(n+2)/2 =   9 OK )  squares= 3 (n-1)  +carry=0   [0s]
  n=5  none=  113  disjoint= 14 ((n-1)(n+2)/2 =  14 OK )  squares= 4 (n-1)  +carry=0   [0s]
  n=6  none=  289  disjoint= 20 ((n-1)(n+2)/2 =  20 OK )  squares= 5 (n-1)  +carry=0   [0s]
  n=7  none=  705  disjoint= 27 ((n-1)(n+2)/2 =  27 OK )  squares= 6 (n-1)  +carry=0   [1s]
  n=8  none= 1665  disjoint= 35 ((n-1)(n+2)/2 =  35 OK )  squares= 7 (n-1)  +carry=0   [6s]

  (n=9, 10 were run separately and give 44 and 54, matching (n-1)(n+2)/2.)


## 6. The lower bound — an explicit witness

For each pair $i<j$ define a 1-cochain supported on generator-$i$ edges only:

$$\varphi_{ij}\bigl(x\to x+2^i\bigr)=\bigl[\ x_i=x_{i+1}=\cdots=x_{j-1}=1\ \bigr],$$

the indicator that the $+2^i$ carry chain at $x$ reaches bit $j$ — precisely the configurations
the locality predicate refuses to declare.

**Claim A. $\varphi_{ij}$ vanishes on every declared square.** Take the square for $(k,l)$,
$k<l$, at a vertex $x$ where the moves have disjoint support, so the $+2^k$ carry does not reach
bit $l$. Its boundary is $e_k(x)+e_l(x+2^k)-e_l(x)-e_k(x+2^l)$. Three cases:

- $i\notin\{k,l\}$: every term lies on another generator, so all are $0$.
- $i=k$: the value is $[x_i..x_{j-1}\text{ all }1]-[(x+2^l)_i..(x+2^l)_{j-1}\text{ all }1]$. If
  $l\ge j$, adding $2^l$ leaves bits $i..j-1$ alone and the terms cancel. If $i<l<j$,
  disjointness says bits $i..l-1$ of $x$ are not all 1; adding $2^l$ cannot change those bits,
  so both indicators are $0$.
- $i=l$: disjointness says the $+2^k$ carry never reaches bit $l=i$, so bits $\ge i$ are
  untouched by $+2^k$ and the terms cancel. $\;\blacksquare$

**Claim B. The $\varphi_{ij}$ are independent modulo everything the all-squares complex sees.**
A cochain kills every square exactly when it lies in $Z^1$ of the all-squares complex, so by
Claim A the entire defect of $\varphi_{ij}$ lives on the *undeclared* squares. Evaluate each
$\varphi_{ij}$ on the undeclared $(k,l)$-square at the canonical vertex $x^*_{kl}=2^l-2^k$. The
columns are exactly

$$\mathrm{col}(k,l)=e_{(l,l+1)}-e_{(k,l+1)}\ \ (l\le n-2),\qquad
\mathrm{col}(k,n-1)=2\!\!\sum_{j>k}\!e_{(k,j)} .$$

Group rows by upper index, $R_j=\operatorname{span}\{e_{(i,j)}:i<j\}$, $\dim R_j=j$. Every column
with $l\le n-2$ lies in $R_{l+1}$, and for fixed $j=l+1$ the $j-1$ columns
$\{e_{(j-1,j)}-e_{(k,j)}\}$ span the sum-zero subspace of $R_j$: total rank $\binom{n-1}2$.
Quotient by those; $R_1$ is one-dimensional and untouched, each $R_j$ collapses to its
coordinate sum $\sigma_j$, and the remaining columns map to
$2(\underbrace{0,\dots,0}_{j\le k},1,\dots,1)$ for $k=0,\dots,n-2$ — triangular, hence
independent. Total rank $\binom{n-1}2+(n-1)=\binom n2$. $\;\blacksquare$

$$\dim(B_A/B_D)\ \ge\ \binom n2\quad\text{for all }n.$$

In [5]:
print("== Claim A and Claim B ==")
for n in range(2,9):
    N, eidx, elist = arena(n); E = len(elist)
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    phis = {p: phi(n, p[0], p[1], eidx, elist) for p in pairs}
    dec, und = [], []
    for x in range(N):
        for (i,j) in pairs: (dec if disjoint(x,i,j,n) else und).append((x,i,j))
    viol = 0
    for (x,i,j) in dec:
        d = S_(n,eidx,x,i,j)
        viol += sum(sum(phis[p][k]*c for k,c in d.items()) != 0 for p in pairs)
    D = Sub()
    rows = []
    for b,p in enumerate(pairs):
        row = np.zeros(len(und), dtype=np.int64)
        for a,(x,i,j) in enumerate(und):
            d = S_(n,eidx,x,i,j)
            row[a] = sum(phis[p][k]*c for k,c in d.items()) % P
        rows.append(row)
    for r in rows: D.add(r)
    print(f"  n={n}: declared={len(dec):5d} undeclared={len(und):4d} | ClaimA violations={viol}"
          f" | ClaimB rank={len(D.piv)}/{len(pairs)} "
          f"{'OK ' if viol==0 and len(D.piv)==len(pairs) else 'FAIL'}")

print("\n== the exact column structure the proof uses ==")
for n in range(3,9):
    N, eidx, elist = arena(n)
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    phis = {p: phi(n, p[0], p[1], eidx, elist) for p in pairs}
    bad = 0
    for (k,l) in pairs:
        x = ((1<<l) - (1<<k)) % (1<<n)
        assert not disjoint(x,k,l,n)
        d = S_(n,eidx,x,k,l)
        col = np.array([sum(phis[p][kk]*c for kk,c in d.items()) for p in pairs])
        want = np.zeros(len(pairs), dtype=np.int64)
        if l <= n-2: want[pidx[(l,l+1)]] += 1; want[pidx[(k,l+1)]] -= 1
        else:
            for j in range(k+1,n): want[pidx[(k,j)]] += 2
        bad += not np.array_equal(col, want)
    print(f"  n={n}: {len(pairs):3d} columns, {bad} mismatches against the claimed formula")

print("\n== triangularity: the pairing matrix is invertible at every width ==")
def pairing(n):
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    M = np.zeros((len(pairs),len(pairs)), dtype=np.int64)
    for (k,l) in pairs:
        c = pidx[(k,l)]
        if l <= n-2: M[pidx[(l,l+1)],c] += 1; M[pidx[(k,l+1)],c] -= 1
        else:
            for j in range(k+1,n): M[pidx[(k,j)],c] += 2
    return M
for n in (4,6,8,12,20,30,40):
    M = pairing(n); r = np.linalg.matrix_rank(M.astype(float))
    print(f"  n={n:2d}  {M.shape[0]:4d}x{M.shape[0]:<4d} rank {r:4d} "
          f"{'OK ' if r==M.shape[0] else 'FAIL'}")

== Claim A and Claim B ==
  n=2: declared=    2 undeclared=   2 | ClaimA violations=0 | ClaimB rank=1/1 OK 
  n=3: declared=   14 undeclared=  10 | ClaimA violations=0 | ClaimB rank=3/3 OK 
  n=4: declared=   62 undeclared=  34 | ClaimA violations=0 | ClaimB rank=6/6 OK 
  n=5: declared=  222 undeclared=  98 | ClaimA violations=0 | ClaimB rank=10/10 OK 
  n=6: declared=  702 undeclared= 258 | ClaimA violations=0 | ClaimB rank=15/15 OK 
  n=7: declared= 2046 undeclared= 642 | ClaimA violations=0 | ClaimB rank=21/21 OK 
  n=8: declared= 5630 undeclared=1538 | ClaimA violations=0 | ClaimB rank=28/28 OK 

== the exact column structure the proof uses ==
  n=3:   3 columns, 0 mismatches against the claimed formula
  n=4:   6 columns, 0 mismatches against the claimed formula
  n=5:  10 columns, 0 mismatches against the claimed formula
  n=6:  15 columns, 0 mismatches against the claimed formula
  n=7:  21 columns, 0 mismatches against the claimed formula
  n=8:  28 columns, 0 mismatches again

## 7. The normal form and the upper bound

Write $c_{kl}=[r_{kl}(2^l-2^k)]$ and let $m(x)$ be the number of consecutive 1s of $x$ starting
at bit $j+1$.

> **Normal form.** For every undeclared square,
> $$\bigl[r_{ij}(x)\bigr]=(-1)^{x_j}\Bigl(c_{ij}+\sum_{t=1}^{m(x)}\bigl(c_{i,j+t}-c_{j,j+t}\bigr)\Bigr)
> \;=\;(-1)^{x_j}Q_{ij}^{m(x)} .$$

An exponentially large family of basepoint-labelled squares factors through two data: the seam
bit and the carry-run depth. Nothing else about $x$ is projected.

### 7.1 L1 — transport below the seam

**Claim.** The class does not depend on any bit of $x$ below $i$.

**Proof.** Let $\ell=x\bmod2^i\neq0$, let $k$ be the position of its lowest set bit, and put
$y=x-2^k$, so $y_k=0$. The cube identity on $k<i<j$ reads

$$r_{ij}(y+2^k)-r_{ij}(y)=\bigl[r_{kj}(y+2^i)-r_{kj}(y)\bigr]-\bigl[r_{ki}(y+2^j)-r_{ki}(y)\bigr].$$

Every face on the right is declared: $r_{ki}$ is undeclared only when bits $k..i-1$ are all 1
and $y_k=0$, $r_{kj}$ only when bits $k..j-1$ are all 1 which again needs $y_k=1$, and adding
$2^i$ or $2^j$ changes nothing below $i$, so all four bases keep $y_k=0$. Thus
$[r_{ij}(x)]=[r_{ij}(y)]$, and the step strictly decreases $\ell$. $\;\blacksquare$

### 7.2 L2 at the top — exact, not modulo anything

In $\mathbb Z/2^n$ the generator $2^{n-1}$ is an involution, so $\{y,y+2^{n-1}\}$ is the same
edge from either side and $e_{n-1}(y+2^{n-1})=-e_{n-1}(y)$. The two squares use the same four
edges, and substituting term by term gives
$\partial r_{i,n-1}(x)+\partial r_{i,n-1}(x+2^{n-1})=0$ **in the chain group**.

**Base case.** With L1 clearing every bit below $i$, bits $i..n-2$ pinned to 1 by the undeclared
condition, and bit $n-1$ the only remaining freedom, this gives
$[r_{i,n-1}(x)]=(-1)^{x_{n-1}}c_{i,n-1}$ with no further input.

### 7.3 L2 below the top is the carry triangle

Stack $r_{ij}(x)$ and $r_{ij}(x\oplus2^j)$ for $x_j=0$. They share an edge, and the domino they
form has the boundary of the next pair's square — up to two copies of the carry triangle:

$$\partial r_{ij}(x)+\partial r_{ij}(x\oplus2^j)=\partial r_{i,j+1}(x)+T_j(x+2^i)-T_j(x),
\qquad T_j(y)=e_j(y)+e_j(y+2^j)-e_{j+1}(y),$$

an exact 1-chain identity. So the sign in the normal form is not extra data: it *is* the
undeclared carry identity $A_j^2=A_{j+1}$, seen from inside the quotient. L2 reduces to one
membership, $\partial r_{i,j+1}(z)+T_j(z+2^i)-T_j(z)\in B_D$ for $z_j=0$.

### 7.4 The staircase closes it

Let $z$ have bits $i..j-1$ all 1 and $z_j=0$. Set $u_j=z$ and $u_k=u_{k+1}-2^k$ for
$k=j-1,\dots,i$ — each step clears a bit that is set, so no borrow occurs — and put
$w=u_i=z-(2^j-2^i)$, so $w+2^j=z+2^i$. The single carry edge $z\to z+2^i$ is replaced by a
staircase that clears the occupied bits one at a time and then makes the $2^j$ jump. Two exact
identities drive it:

$$\textbf{(A)}\ \ T_j(y+2^k)-T_j(y)=S_{kj}(y)+S_{kj}(y+2^j)-S_{k,j+1}(y),\qquad k<j\le n-2$$
$$\textbf{(B)}\ \ T_j(w+2^j)-T_j(w)=-S_{j,j+1}(w)$$

Telescoping (A) along the staircase and applying (B) for the final jump,

$$\partial r_{i,j+1}(z)+T_j(z+2^i)-T_j(z)=S_{i,j+1}(z)-S_{j,j+1}(w)
-\sum_{k=i}^{j-1}\bigl[S_{kj}(u_k)+S_{kj}(u_k+2^j)-S_{k,j+1}(u_k)\bigr],$$

and every term is declared, by three bit facts: $u_k$ has bit $k$ cleared and adding $2^j$ never
touches bit $k$; $w_j=0$; $z_j=0$. No induction on width. $\;\blacksquare$

### 7.5 Assembling

With L1, the base case, L2, and the telescoping recursion
$Q_{ij}^m=c_{ij}+Q_{i,j+1}^{m-1}-Q_{j,j+1}^{m-1}$ — whose geometric source is the cube identity
at $k=j+1$, so the induction runs downward from $j=n-1$ — the normal form is a theorem. Hence
$B_A/B_D=\operatorname{span}\{c_{ij}\}$, so $\dim\le\binom n2$; the witness gives $\ge$; the two
meet.

Identities (A) and (B) are checked below twice: in the arena, and **formally** in symbols
$(\text{generator},\text{offset})$ with offsets left as unreduced integers — no modulus and no
$n$ anywhere.

In [6]:
print("== L1: the clearing step ==")
for n in range(3,9):
    N, eidx, E, pairs, und, BD = build(n)
    steps = free = same = 0
    for (x,i,j) in und:
        low = x & ((1<<i)-1)
        if low == 0: continue
        k = (low & -low).bit_length()-1; y = x - (1<<k); steps += 1
        sides = [(y,k,j), ((y+(1<<i))%N,k,j), (y,k,i), ((y+(1<<j))%N,k,i)]
        free += all(disjoint(b,a,c,n) for (b,a,c) in sides)
        same += not np.any(BD.reduce((vec(S_(n,eidx,x,i,j),E) - vec(S_(n,eidx,y,i,j),E)) % P))
    print(f"  n={n}: {steps:5d} steps | four side faces declared: {free} | class unchanged: "
          f"{same}  {'OK ' if free==steps==same else 'FAIL'}")

print("\n== L2 at j = n-1: exact in the chain group ==")
for n in range(3,10):
    N, eidx, elist = arena(n); E = len(elist); bad = tot = 0
    for x in range(N):
        for i in range(n-1):
            tot += 1
            bad += bool(np.any((vec(S_(n,eidx,x,i,n-1),E)
                              + vec(S_(n,eidx,(x+(1<<(n-1)))%N,i,n-1),E)) % P))
    print(f"  n={n}: {tot:5d} squares, {bad} failures")

print("\n== the domino identity, exact, for x_j = 0 ==")
for n in range(3,8):
    N, eidx, elist = arena(n); E = len(elist); bad = tot = 0
    for x in range(N):
        for (i,j) in itertools.combinations(range(n),2):
            if j > n-2 or (x>>j)&1: continue
            tot += 1; d = {}
            addd(d, S_(n,eidx,x,i,j)); addd(d, S_(n,eidx,x^(1<<j),i,j))
            addd(d, S_(n,eidx,x,i,j+1), -1)
            addd(d, T_(n,eidx,(x+(1<<i))%N,j), -1); addd(d, T_(n,eidx,x,j))
            bad += bool(np.any(vec(d,E)))
    print(f"  n={n}: {tot:5d} instances, {bad} failures")

== L1: the clearing step ==
  n=3:     2 steps | four side faces declared: 2 | class unchanged: 2  OK 
  n=4:    12 steps | four side faces declared: 12 | class unchanged: 12  OK 
  n=5:    46 steps | four side faces declared: 46 | class unchanged: 46  OK 
  n=6:   144 steps | four side faces declared: 144 | class unchanged: 144  OK 
  n=7:   402 steps | four side faces declared: 402 | class unchanged: 402  OK 
  n=8:  1044 steps | four side faces declared: 1044 | class unchanged: 1044  OK 

== L2 at j = n-1: exact in the chain group ==
  n=3:    16 squares, 0 failures
  n=4:    48 squares, 0 failures
  n=5:   128 squares, 0 failures
  n=6:   320 squares, 0 failures
  n=7:   768 squares, 0 failures
  n=8:  1792 squares, 0 failures
  n=9:  4096 squares, 0 failures

== the domino identity, exact, for x_j = 0 ==
  n=3:     4 instances, 0 failures
  n=4:    24 instances, 0 failures
  n=5:    96 instances, 0 failures
  n=6:   320 instances, 0 failures
  n=7:   960 instances, 0 failures


In [7]:
# ---------- (A) and (B) formally: no modulus, no n, offsets unreduced
def e_(g,o): return {(g,o):1}
def a_(a,b,s=1):
    for k,v in b.items(): a[k] = a.get(k,0)+s*v
    return a
def nz_(d): return any(v != 0 for v in d.values())
def Tf(j,y):
    d={}; a_(d,e_(j,y)); a_(d,e_(j,y+2**j)); a_(d,e_(j+1,y),-1); return d
def Sf(p,q,y):
    d={}; a_(d,e_(p,y)); a_(d,e_(q,y+2**p)); a_(d,e_(q,y),-1); a_(d,e_(p,y+2**q),-1); return d

print("== (A) and (B) as FORMAL chain identities ==")
bad = tot = 0
for j in range(1,10):
    for k in range(j):
        for y in (0,5,37,1000,2**12+3,2**20+2**7):
            tot += 1; d = {}
            a_(d,Tf(j,y+2**k)); a_(d,Tf(j,y),-1)
            a_(d,Sf(k,j,y),-1); a_(d,Sf(k,j,y+2**j),-1); a_(d,Sf(k,j+1,y))
            bad += nz_(d)
print(f"  (A): {tot} formal instances, {bad} nonzero residues")
bad = tot = 0
for j in range(0,10):
    for w in (0,5,37,1000,2**12+3,2**20+2**7):
        tot += 1; d = {}
        a_(d,Tf(j,w+2**j)); a_(d,Tf(j,w),-1); a_(d,Sf(j,j+1,w))
        bad += nz_(d)
print(f"  (B): {tot} formal instances, {bad} nonzero residues")

print("\n== the whole staircase, formal, and every term declared ==")
bad = tot = dec_bad = dec_tot = 0
def undec(p,q,y): return all((y>>t)&1 for t in range(p,q))
for j in range(1,10):
    for i in range(j):
        for hi in (0, 2**(j+2), 2**(j+3)+2**(j+5), 2**(j+1)+2**(j+2)+2**(j+9)):
            z = hi + (2**j - 2**i); tot += 1
            lhs = {}; a_(lhs,Sf(i,j+1,z)); a_(lhs,Tf(j,z+2**i)); a_(lhs,Tf(j,z),-1)
            u = {j:z}
            for kk in range(j-1,i-1,-1): u[kk] = u[kk+1] - 2**kk
            w = u[i]; assert w + 2**j == z + 2**i
            terms = [(i,j+1,z,+1), (j,j+1,w,-1)]
            for kk in range(i,j):
                terms += [(kk,j,u[kk],-1), (kk,j,u[kk]+2**j,-1), (kk,j+1,u[kk],+1)]
            rhs = {}
            for (p,q,y,s) in terms: a_(rhs, Sf(p,q,y), s)
            d = {}; a_(d,lhs); a_(d,rhs,-1); bad += nz_(d)
            for (p,q,y,s) in terms: dec_tot += 1; dec_bad += undec(p,q,y)
print(f"  staircase: {tot} formal instances, {bad} nonzero residues")
print(f"  declaredness: {dec_tot} terms, {dec_bad} undeclared")

print("\n== the staircase in the arena, where 2^(n-1) IS an involution ==")
for n in range(3,11):
    N, eidx, elist = arena(n); E = len(elist); tot = exact = alldec = 0
    for (i,j) in itertools.combinations(range(n),2):
        if j > n-2: continue
        mask = ((1<<(j-i))-1) << i
        for z in range(N):
            if (z & mask) != mask or (z>>j)&1: continue
            tot += 1
            lhs = {}
            addd(lhs, S_(n,eidx,z,i,j+1)); addd(lhs, T_(n,eidx,(z+(1<<i))%N,j))
            addd(lhs, T_(n,eidx,z,j), -1)
            u = {j:z}
            for kk in range(j-1,i-1,-1): u[kk] = u[kk+1] - (1<<kk)
            w = u[i]
            terms = [(z,i,j+1,+1), (w,j,j+1,-1)]
            for kk in range(i,j):
                terms += [(u[kk],kk,j,-1), ((u[kk]+(1<<j))%N,kk,j,-1), (u[kk],kk,j+1,+1)]
            rhs = {}
            for (b,p,q,s) in terms: addd(rhs, S_(n,eidx,b,p,q), s)
            d = {}; addd(d,lhs); addd(d,rhs,-1)
            exact += not np.any(vec(d,E))
            alldec += all(disjoint(b,p,q,n) for (b,p,q,s) in terms)
    print(f"  n={n:2d}: {tot:5d} instances | exact: {exact} | all declared: {alldec}  "
          f"{'OK ' if exact==alldec==tot else 'FAIL'}")

== (A) and (B) as FORMAL chain identities ==
  (A): 270 formal instances, 0 nonzero residues
  (B): 60 formal instances, 0 nonzero residues

== the whole staircase, formal, and every term declared ==
  staircase: 180 formal instances, 0 nonzero residues
  declaredness: 2340 terms, 0 undeclared

== the staircase in the arena, where 2^(n-1) IS an involution ==
  n= 3:     2 instances | exact: 2 | all declared: 2  OK 
  n= 4:    10 instances | exact: 10 | all declared: 10  OK 
  n= 5:    34 instances | exact: 34 | all declared: 34  OK 
  n= 6:    98 instances | exact: 98 | all declared: 98  OK 
  n= 7:   258 instances | exact: 258 | all declared: 258  OK 
  n= 8:   642 instances | exact: 642 | all declared: 642  OK 
  n= 9:  1538 instances | exact: 1538 | all declared: 1538  OK 
  n=10:  3586 instances | exact: 3586 | all declared: 3586  OK 


In [8]:
print("== the normal form, on EVERY undeclared square ==")
for n in range(2,9):
    N, eidx, E, pairs, und, BD = build(n)
    Cc = {p: vec(S_(n,eidx,((1<<p[1])-(1<<p[0]))%N,*p), E) for p in pairs}
    bad = 0
    for (x,i,j) in und:
        pred = Cc[(i,j)].copy()
        for t in range(1, m_of(x,j,n)+1): pred = (pred + Cc[(i,j+t)] - Cc[(j,j+t)]) % P
        if (x>>j)&1: pred = (-pred) % P
        bad += bool(np.any(BD.reduce((vec(S_(n,eidx,x,i,j),E) - pred) % P)))
    print(f"  n={n}: {len(und):5d} undeclared squares, {bad} deviations "
          f"{'OK ' if bad==0 else 'FAIL'}")

print("\n== a single prime is not a rational check: exactly over Q ==")
class SubQ:
    def __init__(s): s.piv = {}
    def reduce(s, v):
        v = list(v)
        for c in sorted(s.piv):
            if v[c]:
                f, row = v[c], s.piv[c]
                v = [a - f*b for a,b in zip(v,row)]
        return v
    def add(s, v):
        v = s.reduce(v); nzr = [k for k,a in enumerate(v) if a]
        if not nzr: return
        c = nzr[0]; inv = Fraction(1,1)/v[c]; v = [a*inv for a in v]
        for d in list(s.piv):
            if s.piv[d][c]:
                f = s.piv[d][c]
                s.piv[d] = [a - f*b for a,b in zip(s.piv[d], v)]
        s.piv[c] = v
def vecQ(d, E):
    v = [Fraction(0)]*E
    for k,c in d.items(): v[k] = Fraction(c)
    return v
for n in range(2,7):
    N, eidx, elist = arena(n); E = len(elist)
    pairs = list(itertools.combinations(range(n),2))
    BDq, und = SubQ(), []
    for x in range(N):
        for (i,j) in pairs:
            if disjoint(x,i,j,n): BDq.add(vecQ(S_(n,eidx,x,i,j), E))
            else: und.append((x,i,j))
    Cq = {p: vecQ(S_(n,eidx,((1<<p[1])-(1<<p[0]))%N,*p), E) for p in pairs}
    bad = 0
    for (x,i,j) in und:
        pred = list(Cq[(i,j)])
        for t in range(1, m_of(x,j,n)+1):
            pred = [a+b-c for a,b,c in zip(pred, Cq[(i,j+t)], Cq[(j,j+t)])]
        if (x>>j)&1: pred = [-a for a in pred]
        diff = [a-b for a,b in zip(vecQ(S_(n,eidx,x,i,j),E), pred)]
        bad += any(BDq.reduce(diff))
    print(f"  n={n}: {len(und):4d} undeclared squares, {bad} deviations over Q "
          f"{'OK ' if bad==0 else 'FAIL'}")

print("\n== the two bounds meet ==")
for n in range(2,9):
    N, eidx, E, pairs, und, BD = build(n)
    dBD = len(BD.piv)
    BA = Sub()
    for v in BD.piv.values(): BA.add(v.copy())
    for (x,i,j) in und: BA.add(vec(S_(n,eidx,x,i,j), E))
    q = len(BA.piv) - dBD; b = E - N + 1 - dBD
    print(f"  n={n}: dim(B_A/B_D)={q:3d} = C(n,2)={len(pairs):3d}  ->  "
          f"b1 = {b:3d} = (n-1)(n+2)/2 = {(n-1)*(n+2)//2:3d}  "
          f"{'OK ' if q==len(pairs) and b==(n-1)*(n+2)//2 else 'FAIL'}")

== the normal form, on EVERY undeclared square ==
  n=2:     2 undeclared squares, 0 deviations OK 
  n=3:    10 undeclared squares, 0 deviations OK 
  n=4:    34 undeclared squares, 0 deviations OK 
  n=5:    98 undeclared squares, 0 deviations OK 
  n=6:   258 undeclared squares, 0 deviations OK 
  n=7:   642 undeclared squares, 0 deviations OK 
  n=8:  1538 undeclared squares, 0 deviations OK 

== a single prime is not a rational check: exactly over Q ==
  n=2:    2 undeclared squares, 0 deviations over Q OK 
  n=3:   10 undeclared squares, 0 deviations over Q OK 
  n=4:   34 undeclared squares, 0 deviations over Q OK 
  n=5:   98 undeclared squares, 0 deviations over Q OK 
  n=6:  258 undeclared squares, 0 deviations over Q OK 

== the two bounds meet ==
  n=2: dim(B_A/B_D)=  1 = C(n,2)=  1  ->  b1 =   2 = (n-1)(n+2)/2 =   2  OK 
  n=3: dim(B_A/B_D)=  3 = C(n,2)=  3  ->  b1 =   5 = (n-1)(n+2)/2 =   5  OK 
  n=4: dim(B_A/B_D)=  6 = C(n,2)=  6  ->  b1 =   9 = (n-1)(n+2)/2 =   9  OK 


## 8. The pair-module dimension

$M_{ij}$, the span of all $(i,j)$-square classes in $B_A/B_D$, is
$\operatorname{span}\{Q_{ij}^m\}_{m=0}^{\,n-1-j}$ by the normal form, and

$$Q_{ij}^{m}-Q_{ij}^{m-1}=c_{i,j+m}-c_{j,j+m}.$$

So the change of basis from $\{c_{ij},\ c_{i,j+1}-c_{j,j+1},\ \dots,\ c_{i,n-1}-c_{j,n-1}\}$ to
$\{Q_{ij}^0,\dots,Q_{ij}^{n-1-j}\}$ is unit triangular. The first family is independent because
each difference carries the generator $c_{i,j+m}$, which occurs in no other member, and the
$c_{ab}$ are a basis of $B_A/B_D$ — there are $\binom n2$ of them spanning a
$\binom n2$-dimensional space. Hence

$$\boxed{\;\dim M_{ij}=n-j\;}$$

and the geometric reading is exact: **the pair-module's dimension is the number of possible
carry-run depths above the seam.**

In [9]:
def Qsym(i,j,m):
    d = {(i,j):1}
    for t in range(1,m+1):
        d[(i,j+t)] = d.get((i,j+t),0)+1; d[(j,j+t)] = d.get((j,j+t),0)-1
    return {k:v for k,v in d.items() if v}

print("== rank of {Q^0 .. Q^(n-1-j)} in the c-basis ==")
bad = 0
for n in range(3,13):
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    for (i,j) in pairs:
        cols = []
        for m in range(n-j):
            v = np.zeros(len(pairs), dtype=np.int64)
            for k,c in Qsym(i,j,m).items(): v[pidx[k]] = c
            cols.append(v)
        if np.linalg.matrix_rank(np.stack(cols,axis=1).astype(float)) != n-j: bad += 1
print(f"  every pair at every width n=3..12: rank == n-j, {bad} failures")

print("\n== against the measured pair-module dimension ==")
for n in (5,6,7):
    N, eidx, E, pairs, und, BD = build(n)
    ok = []
    for (i,j) in pairs:
        Sx = Sub()
        for v in BD.piv.values(): Sx.add(v.copy())
        base = len(Sx.piv)
        for (x,a,b) in und:
            if (a,b)==(i,j): Sx.add(vec(S_(n,eidx,x,i,j), E))
        ok.append(len(Sx.piv)-base == n-j)
    print(f"  n={n}: measured dim M_ij == n-j for {sum(ok)}/{len(pairs)} pairs "
          f"{'OK ' if all(ok) else 'FAIL'}")

== rank of {Q^0 .. Q^(n-1-j)} in the c-basis ==
  every pair at every width n=3..12: rank == n-j, 0 failures

== against the measured pair-module dimension ==
  n=5: measured dim M_ij == n-j for 10/10 pairs OK 
  n=6: measured dim M_ij == n-j for 15/15 pairs OK 
  n=7: measured dim M_ij == n-j for 21/21 pairs OK 


## 9. What the pair of theorems says

$$\dim(B_A/B_D)=\binom n2\qquad\text{vs}\qquad \log_2 F(S)$$

The first is a **rank**: additive over pair directions, identical for every reader of the arena.
The second is a **state count**: multiplicative over the runs of a cut, different for every
reading order. They are two quotient constructions on one transition system, not two
measurements of a hidden quantity, and no rescaling relates them. At $n=7$ the arena has 27
unresolved relation directions while an LSB-first reader carries one bit; the same arena read
even-odd carries $\log_2 36\approx5.2$ bits.

There is a second, sharper obstruction to identifying them: $2L+1$ is odd, so any cut containing
an interior run has memory that is not a power of two. Its retained state has no
bit-decomposition, and there is no register to count.

### The shared sufficient statistic

| static (object) | dynamic (reader) |
|---|---|
| an undeclared $(i,j)$-square keeps seam bit $x_j$ and carry-run depth $m(x)$ | a run keeps first-nontransparent depth $d$ and carry mode |
| $2^n$ basepoints collapse to $(x_j,m)$ | $4^{|S|}$ assignments collapse to a signature tuple |
| $\dim M_{ij}=n-j$ = carry depths above the seam | $2L+1$ = states of an interior run of length $L$ |
| residual identical for every reader | state count depends on the cut $S$ |

Both proofs turned on the same object from opposite ends. The exponentially large coordinate
label is, in each case, the reader's own coordinates rather than the object's structure.

### The carry bit

The carry recurrence $c_{q+1}=g_q\vee(p_qc_q)$ is intrinsic to binary addition. That **one bit**
of retained state suffices is intrinsic to the bottom-anchored cut: the lower carry port is
pinned at $c_0=0$, so every output inside the run is immediately emitted and nothing is owed;
the only survivor is where the carry goes next. Move the anchor and the state stops being a bit.

$$\boxed{\;\text{A system need not retain its history. It must retain exactly the cut-defined
continuation structure through which future inputs can still make past distinctions matter.}\;}$$

## 10. Corrections log

Every item here was caught by a disagreement between two readings, not by inspection.

1. **Owed-set count.** Predicted $L+1$ owed-sets per interior run (proper prefixes plus empty);
   measured $L$. The empty ledger never occurs in the interior — a partial reader of an interior
   run always owes at least its lowest bit — and the transparent state reuses the longest owed
   prefix with a different carry mode. The product $2L+1$ is unchanged; the factorisation is
   $L\times2+1$.
2. **"Middle-out" names several orders.** At $n=5$, reading 0 before 4 gives $[1,3,5,7,2,1]$ and
   reading 4 before 0 gives $[1,3,5,7,5,1]$. Same peak. Not a discrepancy.
3. **"Memory counts relation orbits under the reader's step" — retired, not weakened.** Orbits
   number $n$; measured memory is one bit at every width.
4. **A self-contradictory module claim.** An earlier draft asked for rank 1 per pair-module *and*
   independence across pairs, then said in the next clause that the submodules overlap heavily.
   The literal reading is false: $\dim M_{ij}=n-j$. The correct target was
   $\dim(B_A/B_D)=\binom n2$, and the overlap is the mechanism.
5. **A single prime is not a rational check.** An early pass ran only at $p=1000003$. Re-checked
   at three primes and exactly over $\mathbb Q$.
6. **"LSB-first is the unique minimum" has an $n=2$ exception.** The notebook's own output
   printed *2 of 2 orders* at $n=2$ while the abstract said "exactly one out of $n!$" flatly.
7. **A basepoint convention, mistaken for a 50% failure rate.** The membership test failed on
   exactly half the instances until based at the seam-zero point: $x\oplus2^j=x-2^j$ when
   $x_j=1$, not $x+2^j$.
8. **A broken formal check reported 100% failure.** The chain accumulator left zero-valued keys
   in a dictionary and `bool(dict)` counted them, so every identity looked violated while the
   arena computation said otherwise. Two readings disagreeing meant one reader was broken, and it
   was the new one. Fixed by testing coefficients rather than keys — and this is the case worth
   keeping, because it nearly produced a live 180/180 refutation of a correct proof.
9. **Wording.** "The carry bit is not a fact about addition" was too strong. The carry recurrence
   is intrinsic to binary addition; what the bottom-anchored cut fixes is the *dimensionality* of
   the retained state.

**Attribution.** The staircase decomposition closing the last homological membership (§7.4) and
the transducer argument closing the reader law (§3) were both supplied in review. They are
verified here, not merely adopted.

## 11. Open

1. **The worst-order closed form.** $4\cdot3^{(n-3)/2}$ / $20\cdot3^{(n-6)/2}$, DP-verified to
   $n=60$, with an exchange sketch on run-length density but no proof. A combinatorial
   optimisation over run patterns, distinct from the factorisation theorem.
2. **Carry-lookahead readers.** The reader here receives one bit position at a time. A reader
   receiving prefix aggregates — generate/propagate summaries — declares different relations, so
   its cut algebra is a different object. The natural test of whether the cut law survives beyond
   positional readers.
3. **Arenas where the two quotients agree.** Characterise systems in which residual rank and
   $\log_2$ of the minimal state count are related by a theorem. The register rung ($b_1=0$ under
   locality) is the trivial baseline; nothing is known between.
4. **How physics selects the predicate.** The locality predicate here is *canonical relative to
   exclusion and locality* — it is Mazurkiewicz independence, and for token systems it computes
   the hard-core configuration space. That a physical dynamics selects it does not follow and is
   not claimed. This is the only question in the programme that has not moved.

---

*v1.0 consolidated — September 2026. Reader engine and arena engine are independent; every
number above is live output from this notebook.*